## Importaciones y carga de datos

Cargamos el dataset crudo desde `data/raw/`.
Este notebook es autocontenido y no depende del notebook de EDA.

In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

df = pd.read_csv('../data/raw/youtoxic_english_1000.csv')
print(f"Dataset cargado: {df.shape}")
df.head()

Dataset cargado: (1000, 15)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\nDont you reckon them 'black lives matter' ba...,True,True,False,False,True,False,False,False,False,False,False,False
3,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False


## 4. Preprocesamiento de texto

Transformamos el texto crudo en una representación limpia y normalizada.
Cada paso elimina ruido que podría confundir al modelo y reduce la dimensionalidad
del vocabulario sin perder información relevante.

### 4.1 Limpieza con expresiones regulares

Eliminamos URLs, menciones, caracteres especiales y números.
Estas cadenas no aportan valor semántico para detectar toxicidad.

> Nota: las comillas y signos de puntuación se eliminan intencionalmente.
> El vectorizador TF-IDF trabaja con palabras individuales, no con estructura sintáctica,
> por lo que estos caracteres no aportan valor al modelo.
> En modelos basados en transformers esta decisión debería revisarse.

In [5]:
def clean_text(text):
    text = str(text).lower()                          # minúsculas
    text = re.sub(r'http\S+|www\S+', '', text)        # eliminar URLs
    text = re.sub(r'@\w+', '', text)                  # eliminar menciones
    text = re.sub(r'#\w+', '', text)                  # eliminar hashtags
    text = re.sub(r'\d+', '', text)                   # eliminar números
    text = re.sub(r'[^\w\s]', '', text)               # eliminar puntuación
    text = re.sub(r'\s+', ' ', text).strip()          # eliminar espacios extra
    return text

df['text_clean'] = df['Text'].apply(clean_text)

print("Ejemplo de limpieza:")
for original, clean in zip(df['Text'].head(3), df['text_clean'].head(3)):
    print(f"ORIGINAL: {original}")
    print(f"LIMPIO:   {clean}")
    print("-" * 60)

Ejemplo de limpieza:
ORIGINAL: If only people would just take a step back and not make this case about them, because it wasn't about anyone except the two people in that situation.  To lump yourself into this mess and take matters into your own hands makes these kinds of protests selfish and without rational thought and investigation.  The guy in this video is heavily emotional and hyped up and wants to be heard, and when he gets heard he just presses more and more.  He was never out to have a reasonable discussion.  Kudos to the Smerconish for keeping level the whole time and letting Masri make himself out to be a fool.  How dare he and those that tore that city down in protest make this about themselves and to dishonor the entire incident with their own hate.  By the way, since when did police brutality become an epidemic?  I wish everyone would just stop pretending like they were there and they knew EXACTLY what was going on, because there's no measurable amount of people that hones

### 4.2 Eliminación de stopwords

Las stopwords son palabras muy frecuentes que no aportan valor semántico
(artículos, preposiciones, conjunciones). Eliminarlas reduce el ruido
y mejora la capacidad discriminativa del modelo.

In [6]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['text_clean'] = df['text_clean'].apply(remove_stopwords)

print("Ejemplo tras eliminar stopwords:")
for text in df['text_clean'].head(3):
    print(f"- {text}")
    print("-" * 60)

Ejemplo tras eliminar stopwords:
- people would take step back make case wasnt anyone except two people situation lump mess take matters hands makes kinds protests selfish without rational thought investigation guy video heavily emotional hyped wants heard gets heard presses never reasonable discussion kudos smerconish keeping level whole time letting masri make fool dare tore city protest make dishonor entire incident hate way since police brutality become epidemic wish everyone would stop pretending like knew exactly going theres measurable amount people honestly witnessed incident none us clue way whole issue swung grand jury informed trust majority rule right course action let also thank police officers america actually serve protect even youre bit jerk pull respect job know someone many people going pout held accountable actions people hate police need officer two around emergency
------------------------------------------------------------
- law enforcement trained shoot apprehen

### 4.3 Lematización

La lematización reduce cada palabra a su forma base o lema según el diccionario.
Por ejemplo: "running" → "run", "better" → "good".
Es más precisa que el stemming porque tiene en cuenta el contexto morfológico.

In [7]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

df['text_clean'] = df['text_clean'].apply(lemmatize_text)

print("Ejemplo tras lematización:")
for text in df['text_clean'].head(3):
    print(f"- {text}")
    print("-" * 60)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Ejemplo tras lematización:
- people would take step back make case wasnt anyone except two people situation lump mess take matter hand make kind protest selfish without rational thought investigation guy video heavily emotional hyped want heard get heard press never reasonable discussion kudos smerconish keeping level whole time letting masri make fool dare tore city protest make dishonor entire incident hate way since police brutality become epidemic wish everyone would stop pretending like knew exactly going there measurable amount people honestly witnessed incident none u clue way whole issue swung grand jury informed trust majority rule right course action let also thank police officer america actually serve protect even youre bit jerk pull respect job know someone many people going pout held accountable action people hate police need officer two around emergency
------------------------------------------------------------
- law enforcement trained shoot apprehend trained shoot kil

### 4.4 Decisión sobre Stemming

El stemming fue evaluado pero descartado para este problema.
En detección de toxicidad el matiz semántico es crítico: palabras como "harmless"
y "harm" comparten raíz pero tienen significados opuestos.
La lematización sola ofrece suficiente normalización sin sacrificar significado.

El texto procesado final es el resultado de: limpieza → stopwords → lematización.

### 4.5 Guardado del dataset procesado

Guardamos el dataset limpio en `data/processed/` para mantener separados
los datos crudos de los procesados. El archivo crudo nunca se modifica.